# Práctica corta: riesgo de ciberseguridad y regresión lineal

Este cuaderno es una versión reducida del estudio de riesgo. El conjunto de datos contiene **seis columnas** y 400 observaciones sintéticas. El objetivo es revisar estadísticos descriptivos, filtrar filas y columnas, interpretar relaciones y estimar una regresión lineal simple y múltiple.

Varias celdas de código están **incompletas**. Debe completarlas empleando listas, tuplas y diccionarios, además de operaciones de `pandas`.

**Reglas de trabajo**

1. Ejecute las celdas en orden.
2. No modifique el nombre de la variable `df`.
3. Después de cada bloque de código, responda las preguntas en la celda markdown indicada.
4. No interprete `risk_score` como prueba de un incidente; es un índice continuo de prioridad.

## 1. Diccionario de variables

Cada fila representa una ventana de una hora para un equipo.

| Columna | Tipo | Significado |
|---|---|---|
| `device_id` | texto | Identificador anónimo del equipo. No es una magnitud. |
| `failed_logins` | entero | Intentos de autenticación fallidos en la hora. |
| `outbound_mb` | continuo | Volumen de tráfico saliente, en megabytes. |
| `off_hours_ratio` | continuo en [0, 1] | Fracción de actividad fuera del horario habitual. |
| `known_vulnerabilities` | entero | Número de vulnerabilidades conocidas del equipo. |
| `risk_score` | continuo en [0, 100] | Variable respuesta. Valores altos indican mayor prioridad de revisión. |

La variable respuesta es `risk_score`. Las variables explicativas numéricas son las cuatro restantes, excluyendo `device_id`.

## 2. Carga de librerías y generación de datos

Esta celda ya está completa. Ejecútela para crear `df`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
plt.style.use("seaborn-v0_8-whitegrid")

RANDOM_STATE = 42
TARGET = "risk_score"

def generate_short_security_data(n=400, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    failed = rng.poisson(3.2, n)
    outbound = rng.lognormal(3.6, 0.55, n)
    off_hours = rng.beta(2, 5, n)
    vulnerabilities = rng.poisson(1.4, n)
    noise = rng.normal(0, 5, n)
    risk = (
        18
        + 3.1 * failed
        + 0.09 * outbound
        + 22 * off_hours
        + 4.4 * vulnerabilities
        + noise
    )
    return pd.DataFrame({
        "device_id": [f"DEV-{i:03d}" for i in range(1, n + 1)],
        "failed_logins": failed,
        "outbound_mb": outbound,
        "off_hours_ratio": off_hours,
        "known_vulnerabilities": vulnerabilities,
        TARGET: np.clip(risk, 0, 100),
    })

df = generate_short_security_data()
print(f"Filas: {len(df)} | Columnas: {df.shape[1]}")
df.head()

## 3. Práctica con diccionarios y tuplas

Un **diccionario** asocia una clave a un valor. Aquí se utiliza para documentar columnas y para seleccionar variables.

Una **tupla** es una secuencia inmutable. Resulta útil para fijar un conjunto de columnas que no debe modificarse por accidente.

Complete el diccionario `column_glossary` con una frase breve para cada columna. Complete también la tupla `explanatory_columns` con las cuatro variables numéricas que **no** son la respuesta.

In [ ]:
# TODO 1. Complete el glosario. Cada clave debe coincidir con el nombre de una columna.
column_glossary = {
    "device_id": "identificador anónimo del equipo",
    "failed_logins": "",  # escriba aquí el significado
    "outbound_mb": "",
    "off_hours_ratio": "",
    "known_vulnerabilities": "",
    "risk_score": "",
}

# TODO 2. Escriba una tupla con las cuatro variables explicativas numéricas.
# Ejemplo de forma: ("columna_a", "columna_b", "columna_c", "columna_d")
explanatory_columns = ()  # reemplace esta tupla vacía

print("Número de entradas en el glosario:", len(column_glossary))
print("Columnas explicativas:", explanatory_columns)

assert len(column_glossary) == 6, "El glosario debe contener las seis columnas."
assert all(column_glossary.values()), "Complete todas las definiciones del diccionario."
assert isinstance(explanatory_columns, tuple), "explanatory_columns debe ser una tupla."
assert TARGET not in explanatory_columns, "La variable respuesta no debe incluirse en la tupla."
assert "device_id" not in explanatory_columns, "device_id no es una variable numérica para el modelo."
assert set(explanatory_columns) == {
    "failed_logins",
    "outbound_mb",
    "off_hours_ratio",
    "known_vulnerabilities",
}, "La tupla debe contener exactamente las cuatro variables numéricas explicativas."

### Preguntas 3

Escriba las respuestas debajo de cada ítem.

1. ¿Qué diferencia existe entre una lista y una tupla al guardar nombres de columnas?
2. ¿Por qué `device_id` no debe entrar en el modelo lineal?
3. Si una clave del diccionario no coincide exactamente con el nombre de la columna, ¿qué problema aparece al usarlo para seleccionar datos?

## 4. Selección de columnas y revisión descriptiva

Recuerde:

- **mínimo y máximo**: recorrido observado;
- **media**: centro aritmético, sensible a extremos;
- **mediana**: centro robusto;
- **desviación estándar**: dispersión respecto de la media.

Complete la celda para seleccionar las columnas numéricas a partir de la tupla y calcular esos estadísticos.

In [ ]:
# TODO 3. Construya una lista de columnas numéricas: las de explanatory_columns más TARGET.
numeric_columns = []  # por ejemplo: list(...) + [...]

# TODO 4. Seleccione esas columnas y asígnelas a numeric_df.
numeric_df = None  # df[...]

# TODO 5. Obtenga mínimo, máximo, media, mediana y desviación estándar.
# Puede usar numeric_df.describe() o un diccionario de funciones con .agg({...}).
summary = None

summary

### Preguntas 4

1. ¿Qué variable presenta el mayor recorrido (`max - min`)? ¿Eso implica que es la más importante para predecir el riesgo?
2. Compare media y mediana de `outbound_mb`. ¿Qué sugiere una diferencia apreciable?
3. ¿En qué unidades se expresa cada estadístico de `off_hours_ratio`?

## 5. Filtrado de filas

Un filtro conserva únicamente las observaciones que cumplen una condición. Resulta útil para estudiar subgrupos, no para demostrar causalidad.

Complete los filtros. Puede usar `df.query(...)`, máscaras booleanas o `df.loc[...]`.

In [ ]:
# Umbrales de apoyo. Puede modificarlos después de responder las preguntas.
thresholds = {
    "min_failed_logins": 5,
    "min_outbound_mb": 80,
    "min_vulnerabilities": 3,
}

# TODO 6. Filtre filas con failed_logins >= thresholds["min_failed_logins"].
# Sugerencia: df[df["failed_logins"] >= ...]  o  df.query("...")
high_failed = None

# TODO 7. Filtre con DOS condiciones a la vez (use & o query):
# outbound_mb alto Y known_vulnerabilities alto, según el diccionario.
high_traffic_and_vuln = None

# TODO 8. Defina una tupla de columnas para revisar y seleccione esas columnas
# sobre high_traffic_and_vuln.
review_columns = ()  # por ejemplo ("device_id", "outbound_mb", TARGET)
preview = None  # high_traffic_and_vuln.loc[:, list(review_columns)]

print("Observaciones con muchos inicios fallidos:", len(high_failed))
print("Riesgo medio en ese grupo:", high_failed[TARGET].mean())
print("Riesgo medio global:", df[TARGET].mean())
print("Observaciones del filtro conjunto:", len(high_traffic_and_vuln))
preview.head(8)

### Preguntas 5

1. Compare el riesgo medio del grupo con muchos inicios fallidos frente al promedio global.
2. ¿El filtro conjunto mezcla dos variables? ¿Qué problema crea eso si se quiere atribuir el riesgo a una sola de ellas?
3. Reescriba el filtro de `high_failed` usando `df.query`.
4. Cambie un umbral del diccionario `thresholds` y indique si el patrón se mantiene.

## 6. Histogramas y correlación

Complete el bucle para dibujar un histograma por cada columna numérica. Después calcule las correlaciones con `risk_score`.

In [ ]:
# TODO 9. Recorra numeric_columns y dibuje un histograma de cada una.
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, column in zip(axes.flat, numeric_columns):
    ax.hist(df[column], bins=20, color="#4C78A8", edgecolor="white", alpha=0.85)
    ax.axvline(df[column].mean(), color="#C44E52", linewidth=2, label="media")
    ax.axvline(df[column].median(), color="black", linestyle="--", linewidth=1.5, label="mediana")
    ax.set_title(column)
    ax.set_ylabel("frecuencia")
axes[0, 0].legend()
plt.tight_layout()
plt.show()

# TODO 10. Calcule la correlación de Pearson de cada variable numérica con TARGET.
correlations = df[numeric_columns].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
correlations

In [ ]:
# TODO 11. Identifique la variable de mayor correlación absoluta y dibuje el diagrama de dispersión.
most_significant = correlations.index[0]

plt.figure(figsize=(7, 5))
plt.scatter(df[most_significant], df[TARGET], alpha=0.45, s=22)
slope, intercept = np.polyfit(df[most_significant], df[TARGET], 1)
x_line = np.linspace(df[most_significant].min(), df[most_significant].max(), 80)
plt.plot(x_line, intercept + slope * x_line, color="#C44E52", linewidth=2)
plt.xlabel(most_significant)
plt.ylabel(TARGET)
plt.title(f"Dispersión y ajuste lineal preliminar (r = {correlations.iloc[0]:.3f})")
plt.show()

print("Variable de mayor asociación lineal:", most_significant)

### Preguntas 6

1. ¿En qué histogramas la media y la mediana casi coinciden?
2. Ordene las cuatro variables explicativas según el valor absoluto de su correlación con `risk_score`.
3. Interprete el signo de la correlación más fuerte en términos de seguridad.
4. ¿Una correlación próxima a cero implica que la variable carece de utilidad operativa?
5. ¿Por qué una correlación alta no demuestra causalidad?

## 7. Regresión lineal: de la fórmula al modelo

Para una variable:

$$\hat y = b_0 + b_1 x$$

$$b_1=\frac{\sum (x_i-\bar x)(y_i-\bar y)}{\sum (x_i-\bar x)^2}, \qquad b_0=\bar y-b_1\bar x$$

Complete el cálculo manual sobre todas las observaciones y verifique el resultado con `LinearRegression`.

In [ ]:
x = df[most_significant].to_numpy()
y = df[TARGET].to_numpy()

# TODO 12. Calcule b1 y b0 con las fórmulas de mínimos cuadrados.
b1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)
b0 = y.mean() - b1 * x.mean()

print(f"Ecuación manual: {TARGET} = {b0:.3f} + ({b1:.3f} × {most_significant})")

# TODO 13. Ajuste el mismo modelo con scikit-learn y compare coeficientes.
simple_check = LinearRegression().fit(df[[most_significant]], df[TARGET])
print("Coincide con sklearn:", np.allclose([b0, b1], [simple_check.intercept_, simple_check.coef_[0]]))

### Preguntas 7

1. Interprete $b_1$: ¿qué ocurre con `risk_score` si la variable $x$ aumenta en una unidad?
2. ¿Por qué $b_0$ puede no tener interpretación operativa?
3. ¿Qué minimiza el método de mínimos cuadrados?

## 8. Comparación de modelos en datos de prueba

Se comparan tres especificaciones sobre las mismas filas de prueba:

1. promedio de entrenamiento;
2. regresión simple con la variable más correlacionada;
3. regresión múltiple con las cuatro variables de la tupla.

Complete el diccionario `predictions` y el cálculo de métricas.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[list(explanatory_columns)], df[TARGET], test_size=0.25, random_state=RANDOM_STATE
)

simple_model = LinearRegression().fit(X_train[[most_significant]], y_train)
multiple_model = LinearRegression().fit(X_train, y_train)

# TODO 14. Complete el diccionario de predicciones sobre el conjunto de prueba.
predictions = {
    "media": np.full(len(y_test), y_train.mean()),
    "simple": simple_model.predict(X_test[[most_significant]]),
    "multiple": multiple_model.predict(X_test),
}

# TODO 15. Recorra el diccionario y calcule MAE, RMSE y R2.
rows = []
for name, y_pred in predictions.items():
    rows.append((
        name,
        mean_absolute_error(y_test, y_pred),
        np.sqrt(mean_squared_error(y_test, y_pred)),
        r2_score(y_test, y_pred),
    ))

metrics = pd.DataFrame(rows, columns=["model", "MAE", "RMSE", "R2"]).set_index("model")
metrics

In [ ]:
ax = metrics[["MAE", "RMSE"]].plot.bar(figsize=(8, 4), color=["#4C78A8", "#F58518"])
ax.set_ylabel("Error en puntos de risk_score")
ax.set_title("Error de predicción en prueba")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

print("Coeficientes del modelo múltiple")
pd.Series(multiple_model.coef_, index=explanatory_columns, name="coefficient")

### Preguntas 8

1. ¿Qué modelo presenta el menor MAE? Interprete el valor en puntos de `risk_score`.
2. ¿Qué proporción de la variabilidad explica el modelo simple? ¿Y el múltiple?
3. ¿Por qué el RMSE suele ser mayor que el MAE?
4. Interprete el signo de dos coeficientes del modelo múltiple.
5. ¿Añadir variables garantiza siempre una mejora en datos no vistos?

## 9. Predicción sobre un equipo nuevo

Complete el diccionario del nuevo caso y conviértalo en un `DataFrame` con las mismas columnas de la tupla `explanatory_columns`.

In [ ]:
# TODO 16. Defina un equipo hipotético. Modifique los valores y vuelva a ejecutar.
new_device = {
    "failed_logins": 8,
    "outbound_mb": 120.0,
    "off_hours_ratio": 0.55,
    "known_vulnerabilities": 4,
}

# TODO 17. Construya un DataFrame de una fila respetando el orden de explanatory_columns.
new_row = pd.DataFrame([new_device])[list(explanatory_columns)]
predicted = multiple_model.predict(new_row)[0]
new_row.assign(predicted_risk_score=predicted)

### Preguntas 9

1. Antes de ejecutar, formule una hipótesis: si aumentan `failed_logins` y `known_vulnerabilities`, ¿qué espera para `risk_score`?
2. Cambie **una sola** clave del diccionario `new_device` y observe el cambio en la predicción.
3. ¿Por qué esta predicción no debe usarse como prueba de un ataque?

## 10. Entrega breve

En un párrafo, responda:

1. qué columna se asocia de forma más clara con `risk_score`;
2. cómo lo muestran la correlación y la pendiente;
3. si el modelo múltiple mejora al simple en prueba;
4. una limitación del conjunto de datos sintético.

Criterio: la regresión lineal es adecuada cuando la respuesta es continua y la relación puede aproximarse por una recta o un plano. No sustituye la investigación de un incidente real.